# 第 9 讲 · 数据与模型运行

实验沿用 2024 年 `L9.ipynb` 的数据示例和 `L10.ipynb` 的模型示例。
按顺序运行各节，观察文本如何变成模型输入，以及基座权重和 LoRA adapter 如何分别保存与加载。

依赖：`torch`、`transformers`、`tokenizers`、`datasets`、`safetensors`、`peft`。默认用本地样本、手工词表和小型 Llama，安装依赖后可在 CPU 上离线运行。示例的 token ID 与课件中的 Llama 3.1 词表不同。

## 1. 文本、token 与 token ID

`tokenize` 返回 token 字符串，`encode` 返回 ID，直接调用 tokenizer 返回 `input_ids` 和 `attention_mask`。

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

vocab = {token: i for i, token in enumerate([
    "[PAD]", "[UNK]", "[BOS]", "[EOS]", "Tell", "me", "a", "story",
    "about", "Nanjing", "University", ".", "Hello", "world", "!",
])}
backend = Tokenizer(WordLevel(vocab=vocab, unk_token="[UNK]"))
backend.pre_tokenizer = Whitespace()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend, pad_token="[PAD]", unk_token="[UNK]",
    bos_token="[BOS]", eos_token="[EOS]",
    model_input_names=["input_ids", "attention_mask"],
)

sentence = "Tell me a story about Nanjing University."
encoded = tokenizer(sentence)
print(tokenizer.tokenize(sentence))
print(tokenizer.encode(sentence))
print(encoded)
print(tokenizer.decode(encoded["input_ids"]))

## 2. 批处理、padding 与 truncation

比较右侧与左侧 padding。掩码中的 `0` 对应填充位置。`max_length` 与 `truncation=True` 一起限制输入长度。

In [ ]:
texts = [sentence, "Hello world!"]
print(tokenizer(texts)["input_ids"])
for side in ("right", "left"):
    tokenizer.padding_side = side
    batch = tokenizer(texts, padding=True, return_tensors="pt")
    print(side, batch["input_ids"], batch["attention_mask"], sep="\n")

tokenizer.padding_side = "right"
short_batch = tokenizer(
    texts, padding="max_length", max_length=6, truncation=True,
    return_tensors="pt",
)
print(short_batch)
assert short_batch["input_ids"].shape == short_batch["attention_mask"].shape == (2, 6)

## 3. 用 `Dataset.map` 预处理指令数据

本例用与课件 Alpaca 示例相同的字段。指令总是保留，有额外输入时将其拼接在指令后。这里构建的是模型输入，训练标签留到训练流程中再处理。

In [ ]:
from datasets import Dataset

samples = Dataset.from_list([
    {"instruction": sentence, "input": "", "output": "Hello world!"},
    {"instruction": "Tell me a story.", "input": "Nanjing University", "output": "Hello!"},
])
# 联网时可用课件中的数据集替换 samples：
# from datasets import load_dataset
# samples = load_dataset("yahma/alpaca-cleaned", split="train[:100]")

def tokenize_examples(examples):
    prompts = [
        f"{instruction}\n{extra}" if extra else instruction
        for instruction, extra in zip(examples["instruction"], examples["input"])
    ]
    return tokenizer(prompts, truncation=True, max_length=32)

tokenized = samples.map(tokenize_examples, batched=True)
print(tokenized[0])

## 4. 用 `state_dict` 保存与恢复 PyTorch 权重

先创建同结构的模型，再加载参数。检查恢复前后的输出是否一致。文件写入临时目录。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from torch import nn

torch.manual_seed(9)
model = nn.Linear(4, 2)
x = torch.randn(3, 4)
print(model.state_dict().keys())

with TemporaryDirectory() as temp:
    checkpoint = Path(temp) / "model_weights.pt"
    torch.save(model.state_dict(), checkpoint)
    restored = nn.Linear(4, 2)
    restored.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
    with torch.inference_mode():
        torch.testing.assert_close(model(x), restored(x))
print("权重加载后，前向输出一致")

## 5. Hugging Face 基座与 LoRA adapter

用一个随机初始化的小型 Llama 演示 `save_pretrained` 和 `from_pretrained`。基座保存配置和权重，adapter 保存自己的配置与低秩参数。训练一步后，将 adapter 装回同一份基座并对比输出。

In [ ]:
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from safetensors.torch import load_file
from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size=len(tokenizer), hidden_size=32, intermediate_size=64,
    num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,
    max_position_embeddings=64, pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id,
)
inputs = tokenizer(sentence, return_tensors="pt")

with TemporaryDirectory() as temp:
    base_dir = Path(temp) / "base"
    adapter_dir = Path(temp) / "adapter"
    LlamaForCausalLM(config).save_pretrained(base_dir, safe_serialization=True)
    print("基座权重示例", list(load_file(base_dir / "model.safetensors"))[:3])

    base = LlamaForCausalLM.from_pretrained(base_dir)
    adapter = get_peft_model(base, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=8,
        target_modules=["q_proj", "v_proj"],
    ))
    adapter.print_trainable_parameters()
    trainable = [parameter for parameter in adapter.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=0.01)
    adapter.train()
    loss = adapter(**inputs, labels=inputs["input_ids"]).loss
    loss.backward()
    optimizer.step()
    adapter.save_pretrained(adapter_dir, safe_serialization=True)

    restored_base = LlamaForCausalLM.from_pretrained(base_dir)
    restored_adapter = PeftModel.from_pretrained(restored_base, adapter_dir)
    adapter.eval()
    restored_adapter.eval()
    with torch.inference_mode():
        torch.testing.assert_close(
            adapter(**inputs).logits, restored_adapter(**inputs).logits,
        )
print("重新加载基座和 adapter 后，前向输出一致")